# Batch processing of JSON file

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [0]:
schema=StructType([
    StructField("InvoiceNumber", StringType()),
    StructField('CreatedTime',StringType()),
    StructField('StoreID', StringType()),
    StructField('PosID',StringType()),
    StructField('CashierID',StringType()),
    StructField('CustomerType', StringType()),
    StructField('CustomerCardNo', StringType()),
    StructField('TotalAmount', FloatType()),
    StructField('NumberOfItems', IntegerType()),
    StructField('PaymentMethod', StringType()),
    StructField('TaxableAmount', FloatType()),
    StructField('CGST', FloatType()),
    StructField('SGST', FloatType()),
    StructField('CESS',  FloatType()),
    StructField('DeliveryType', StringType()),
    StructField('DeliveryAddress', StructType([
        StructField('AddressLine', StringType()),
        StructField('City', StringType()),
        StructField('State', StringType()),
        StructField('PinCode', StringType()),
        StructField('ContactNumber', StringType())
    ])),
    StructField('InvoiceLineItems', ArrayType(
        StructType([
            StructField('ItemCode', StringType()),
            StructField('ItemDescription', StringType()),
            StructField('ItemPrice', FloatType()),
            StructField('ItemQty',IntegerType()),
            StructField('TotalValue',FloatType())
        ])
    ))
])

In [0]:
df=(
        spark.read
            .format('json')
            .schema(schema)
            .load('/Volumes/cat_spark_streaming/jsonstream/dataset/json-datasets/') 
)

### Exploading JSON with withColumn

In [0]:
flattened_stage1_df=df.withColumn('InvoiceLineItems',explode_outer('InvoiceLineItems'))

flatten_json=(
            flattened_stage1_df
                  .select('InvoiceNumber',
                          'CreatedTime',
                          'StoreID',
                          'PosID',
                          'CashierID',
                          'CustomerType',
                          'CustomerCardNo',
                          'TotalAmount',
                          'NumberOfItems',
                          'PaymentMethod',
                          'TaxableAmount',
                          'CGST',
                          'SGST',
                          'CESS',
                          'DeliveryType',
                          'DeliveryAddress.AddressLine',
                          'DeliveryAddress.City',
                          'DeliveryAddress.State',
                          'DeliveryAddress.PinCode',
                          'DeliveryAddress.ContactNumber',
                          'InvoiceLineItems.ItemCode',
                          'InvoiceLineItems.ItemDescription',
                          'InvoiceLineItems.ItemPrice',
                          'InvoiceLineItems.ItemQty',
                          'InvoiceLineItems.TotalValue'
                     )
)

(
    flatten_json
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('cat_spark_streaming.jsonstream.batchJSON')    
)

### Not using withColumn

In [0]:
flattened_stage2_df = (
    df.select(
        'InvoiceNumber',
        'CreatedTime',
        'StoreID',
        'PosID',
        'CashierID',
        'CustomerType',
        'CustomerCardNo',
        'TotalAmount',
        'NumberOfItems',
        'PaymentMethod',
        'TaxableAmount',
        'CGST',
        'SGST',
        'CESS',
        'DeliveryType',
        'DeliveryAddress.AddressLine',
        'DeliveryAddress.City',
        'DeliveryAddress.State',
        'DeliveryAddress.PinCode',
        'DeliveryAddress.ContactNumber',
        explode_outer(col('InvoiceLineItems')).alias('InvoiceLineItems'))
            .select( '*',
                    'InvoiceLineItems.ItemCode',
                    'InvoiceLineItems.ItemDescription',
                    'InvoiceLineItems.ItemPrice',
                    'InvoiceLineItems.ItemQty',
                    'InvoiceLineItems.TotalValue'
    )
)


(
    flattened_stage2_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('cat_spark_streaming.jsonstream.batchJSONWIthSelect')    
)